# Species Category Usage on English Wikipedia

* Number of species on Wikidata with sitelinks to enwiki (page IDs)
* Number of region-specific categories (QIDs)
* Categories on enwiki species that map to region-specific categories

## Imports / settings / etc.

In [1]:
import wmfdata

In [2]:
spark = wmfdata.spark.create_session(app_name='pyspark large - wikidata categories',
                                     type='yarn-large', # local, yarn-regular, yarn-large
                                    )  

SPARK_HOME: /usr/lib/spark3
Using Hadoop client lib jars at 3.2.0, provided by Spark.
PYSPARK_PYTHON=/opt/conda-analytics/bin/python3


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/07/23 14:26:02 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).
24/07/23 14:26:03 WARN Utils: Service 'sparkDriver' could not bind on port 12000. Attempting port 12001.
24/07/23 14:26:03 WARN Utils: Service 'sparkDriver' could not bind on port 12001. Attempting port 12002.
24/07/23 14:26:03 WARN Utils: Service 'sparkDriver' could not bind on port 12002. Attempting port 12003.
24/07/23 14:26:03 WARN Utils: Service 'sparkDriver' could not bind on port 12003. Attempting port 12004.
24/07/23 14:26:03 WARN Utils: Service 'sparkDriver' could not bind on port 12004. Attempting port 12005.
24/07/23 14:26:03 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/07/23 14:26:03 WARN Utils: Service 'SparkUI' c

In [3]:
print("Wikidata:")
spark.sql('show partitions wmf.wikidata_item_page_link').show(50, False)
print("\nMediawiki:")
spark.sql("show partitions wmf_raw.mediawiki_project_namespace_map").show(50, False)

Wikidata:
+-------------------+
|partition          |
+-------------------+
|snapshot=2024-05-06|
|snapshot=2024-05-20|
|snapshot=2024-05-27|
|snapshot=2024-06-03|
|snapshot=2024-06-10|
|snapshot=2024-06-17|
|snapshot=2024-06-24|
+-------------------+


Mediawiki:
+------------------------+
|partition               |
+------------------------+
|snapshot=2016-12_private|
|snapshot=2017-07_private|
|snapshot=2023-12        |
|snapshot=2024-01        |
|snapshot=2024-02        |
|snapshot=2024-03        |
|snapshot=2024-04        |
|snapshot=2024-05        |
|snapshot=2024-06        |
+------------------------+



In [4]:
mw_snapshot = '2024-05'  # 2020-07 means data is up to 31 July 2020
wd_snapshot = '2024-06-03'  # 2020-07-31 means data is up to 31 July 2020

## Analysis

In [5]:
# value info in wikidata entity table (https://wikitech.wikimedia.org/wiki/Analytics/Data_Lake/Edits/Wikidata_entity)
# is a string as opposed to struct (because it has a variable schema)
# this UDF extracts the QID value (or null if doesn't exist)
def getValue(obj):
    try:
        d =  eval(obj)
        return d.get('id')
    except Exception:
        return None
    
spark.udf.register('getValue', getValue, 'string')

<function __main__.getValue(obj)>

In [9]:
query = f"""
WITH wikis AS (
    SELECT DISTINCT
      database_code
    FROM canonical_data.wikis
    WHERE
      database_group = 'wikipedia'
      AND status = 'open'
      AND visibility = 'public'
      AND editability = 'public'
),
relevant_qids AS (
    SELECT DISTINCT
      item_id AS category_qid
    FROM wmf.wikidata_item_page_link wd
    INNER JOIN wikis wp
      ON (wd.wiki_db = wp.database_code)
    WHERE
      snapshot = '{wd_snapshot}'
      AND page_namespace = 14
      AND wiki_db LIKE '%wiki'
),
exploded_statements AS (
    SELECT
      q.category_qid,
      explode(claims) AS claim
    FROM wmf.wikidata_entity w
    INNER JOIN relevant_qids q
      ON (w.id = q.category_qid)
    WHERE
      w.snapshot = '{wd_snapshot}'
),
category_parts AS (
    SELECT
      category_qid,
      getValue(claim.mainSnak.dataValue.value) AS category_part
    FROM exploded_statements
    WHERE
      claim.mainSnak.property = 'P971'
),
countries AS (
    SELECT DISTINCT
      wikidata_id AS country_item,
      name AS country_name
    FROM canonical_data.countries
)
SELECT DISTINCT
  category_qid,
  country_name
FROM category_parts cp
INNER JOIN countries c
  ON (cp.category_part = c.country_item)
"""

print(query)
result = spark.sql(query)
result.coalesce(1).write.csv(path=f"/user/isaacj/category-countries",
                             compression='gzip', header=True, sep="\t")


WITH wikis AS (
    SELECT DISTINCT
      database_code
    FROM canonical_data.wikis
    WHERE
      database_group = 'wikipedia'
      AND status = 'open'
      AND visibility = 'public'
      AND editability = 'public'
),
relevant_qids AS (
    SELECT DISTINCT
      item_id AS category_qid
    FROM wmf.wikidata_item_page_link wd
    INNER JOIN wikis wp
      ON (wd.wiki_db = wp.database_code)
    WHERE
      snapshot = '2024-06-03'
      AND page_namespace = 14
      AND wiki_db LIKE '%wiki'
),
exploded_statements AS (
    SELECT
      q.category_qid,
      explode(claims) AS claim
    FROM wmf.wikidata_entity w
    INNER JOIN relevant_qids q
      ON (w.id = q.category_qid)
    WHERE
      w.snapshot = '2024-06-03'
),
category_parts AS (
    SELECT
      category_qid,
      getValue(claim.mainSnak.dataValue.value) AS category_part
    FROM exploded_statements
    WHERE
      claim.mainSnak.property = 'P971'
),
countries AS (
    SELECT DISTINCT
      wikidata_id AS country_item,
 

24/07/12 17:22:10 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
24/07/12 17:24:39 ERROR TransportClient: Failed to send RPC RPC 8094992610380189844 to /10.64.5.41:33340: java.nio.channels.ClosedChannelException
java.nio.channels.ClosedChannelException
	at io.netty.channel.AbstractChannel$AbstractUnsafe.newClosedChannelException(AbstractChannel.java:957)
	at io.netty.channel.AbstractChannel$AbstractUnsafe.write(AbstractChannel.java:865)
	at io.netty.channel.DefaultChannelPipeline$HeadContext.write(DefaultChannelPipeline.java:1367)
	at io.netty.channel.AbstractChannelHandlerContext.invokeWrite0(AbstractChannelHandlerContext.java:717)
	at io.netty.channel.AbstractChannelHandlerContext.invokeWriteAndFlush(AbstractChannelHandlerContext.java:764)
	at io.netty.channel.AbstractChannelHandlerContext$WriteTask.run(AbstractChannelHandlerContext.java:1071)
	at io.netty.util.concurrent.AbstractEventEx

In [10]:
!hdfs dfs -ls category-countries

Found 2 items
-rw-r-----   3 isaacj isaacj          0 2024-07-12 17:27 category-countries/_SUCCESS
-rw-r-----   3 isaacj isaacj    2164033 2024-07-12 17:27 category-countries/part-00000-08b4825d-641d-4a66-ab49-c120e2aa8c39-c000.csv.gz


In [12]:
!hdfs dfs -text category-countries/part-00000-08b4825d-641d-4a66-ab49-c120e2aa8c39-c000.csv.gz | head

category_qid	country_name
Q15147507	Australia
Q30649112	Australia
Q25320777	Spain
Q65703267	Denmark
Q8346377	Latvia
Q24925498	Dominican Republic
Q8513754	Bulgaria
Q24889869	Australia
Q60797821	Slovenia
text: Unable to write to output stream.


In [13]:
!hdfs dfs -text category-countries/part-00000-08b4825d-641d-4a66-ab49-c120e2aa8c39-c000.csv.gz | wc -l

340617


In [14]:
!hdfs dfs -copyToLocal category-countries/part-00000-08b4825d-641d-4a66-ab49-c120e2aa8c39-c000.csv.gz category-countries.tsv.gz

## Compute bulk predictions

In [3]:
import pandas as pd

In [5]:
# Create table with Category QID -> Region mapping
spark.createDataFrame(pd.read_csv('category-countries.tsv.gz', sep='\t')).createOrReplaceTempView('cat_qid_to_region')
spark.sql("SELECT * FROM cat_qid_to_region LIMIT 10").show(50, False)

24/07/23 14:28:23 WARN TaskSetManager: Stage 0 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.


+------------+------------------+
|category_qid|country_name      |
+------------+------------------+
|Q15147507   |Australia         |
|Q30649112   |Australia         |
|Q25320777   |Spain             |
|Q65703267   |Denmark           |
|Q8346377    |Latvia            |
|Q24925498   |Dominican Republic|
|Q8513754    |Bulgaria          |
|Q24889869   |Australia         |
|Q60797821   |Slovenia          |
|Q97050848   |Egypt             |
+------------+------------------+



In [7]:
category_results_table = "isaacj.qid_to_country_categories"
create_table_query = f"""
    CREATE TABLE IF NOT EXISTS {category_results_table} (
        pid_from         INT     COMMENT 'Page ID of Wikipedia article -- e.g., 12345',
        wiki_db          STRING  COMMENT 'Wiki -- e.g., enwiki',
        country          STRING  COMMENT 'Region name'
    )
    """

print(create_table_query)
spark.sql(create_table_query)


    CREATE TABLE IF NOT EXISTS isaacj.qid_to_country_categories (
        pid_from         INT     COMMENT 'Page ID of Wikipedia article -- e.g., 12345',
        wiki_db          STRING  COMMENT 'Wiki -- e.g., enwiki',
        country          STRING  COMMENT 'Region name'
    )
    


24/07/23 14:29:48 WARN ResolveSessionCatalog: A Hive serde table will be created as there is no table provider specified. You can set spark.sql.legacy.createHiveTableByDefault to false so that native data source table will be created instead.
24/07/23 14:29:49 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


DataFrame[]

In [8]:
print("Mediawiki snapshots:")
spark.sql("SHOW PARTITIONS wmf_raw.mediawiki_project_namespace_map").show(50, False)

print("\nWikidata snapshots:")
spark.sql("SHOW PARTITIONS wmf.wikidata_item_page_link").show(50, False)

Mediawiki snapshots:
+------------------------+
|partition               |
+------------------------+
|snapshot=2016-12_private|
|snapshot=2017-07_private|
|snapshot=2024-01        |
|snapshot=2024-02        |
|snapshot=2024-03        |
|snapshot=2024-04        |
|snapshot=2024-05        |
|snapshot=2024-06        |
+------------------------+


Wikidata snapshots:
+-------------------+
|partition          |
+-------------------+
|snapshot=2024-05-20|
|snapshot=2024-05-27|
|snapshot=2024-06-03|
|snapshot=2024-06-10|
|snapshot=2024-06-17|
|snapshot=2024-06-24|
|snapshot=2024-07-08|
+-------------------+



In [9]:
mw_snapshot = "2024-06"
wd_snapshot = "2024-07-08"

In [16]:
print_for_hive = False
do_execute = True

query = f"""
WITH relevant_wikis AS (
    SELECT
      DISTINCT(database_code) AS wiki_db
    FROM canonical_data.wikis
    WHERE
      database_group = 'wikipedia'
      AND status = 'open'
      AND visibility = 'public'
      AND editability = 'public'
),
title_to_id AS (
    SELECT page_id,
           page_title,
           mp.wiki_db
      FROM wmf_raw.mediawiki_page mp
     INNER JOIN relevant_wikis wp
           ON (mp.wiki_db = wp.wiki_db)
     WHERE page_namespace = 14
           AND snapshot = '{mw_snapshot}'
),
redirects AS (
    SELECT mr.rd_from AS rd_from,
           tti.page_id AS rd_to,
           mr.wiki_db AS wiki_db
      FROM wmf_raw.mediawiki_redirect mr
     INNER JOIN title_to_id tti
           ON (mr.rd_title = tti.page_title
               AND mr.wiki_db = tti.wiki_db)
     WHERE mr.snapshot = '{mw_snapshot}'
           AND mr.rd_namespace = 14
),
categorylinks_reformatted AS (
    SELECT cl.cl_from AS pid_from,
           tti.page_id AS cl_to,
           cl.wiki_db AS wiki_db
      FROM wmf_raw.mediawiki_categorylinks cl
     INNER JOIN title_to_id tti
           ON (cl.cl_to = tti.page_title
               AND cl.wiki_db = tti.wiki_db)
     WHERE snapshot = '{mw_snapshot}'
),
cat_pid_to_country AS (
    SELECT DISTINCT
      wd.page_id,
      wd.wiki_db,
      country_name AS country
    FROM wmf.wikidata_item_page_link wd
    INNER JOIN relevant_wikis db
      ON (wd.wiki_db = db.wiki_db)
    INNER JOIN cat_qid_to_region gt
      ON (wd.item_id = gt.category_qid)
    WHERE
      snapshot = '{wd_snapshot}'
      AND page_namespace = 14
),
catlinks_redirects_resolved AS (
    SELECT cl.pid_from,
           COALESCE(r.rd_to, cl.cl_to) AS cl_to,
           cl.wiki_db AS wiki_db
      FROM categorylinks_reformatted cl
      LEFT JOIN redirects r
           ON (cl.cl_to = r.rd_from
               AND cl.wiki_db = r.wiki_db)
)
INSERT OVERWRITE TABLE {category_results_table}
SELECT c.pid_from,
       c.wiki_db,
       gt.country
  FROM catlinks_redirects_resolved c
  INNER JOIN cat_pid_to_country gt
       ON (c.cl_to = gt.page_id
           AND c.wiki_db = gt.wiki_db)
"""

if print_for_hive:
    print(re.sub(' +', ' ', re.sub('\n', ' ', query)).strip())
else:
    print(query)

if do_execute:
    result = spark.sql(query)


WITH relevant_wikis AS (
    SELECT
      DISTINCT(database_code) AS wiki_db
    FROM canonical_data.wikis
    WHERE
      database_group = 'wikipedia'
      AND status = 'open'
      AND visibility = 'public'
      AND editability = 'public'
),
title_to_id AS (
    SELECT page_id,
           page_title,
           mp.wiki_db
      FROM wmf_raw.mediawiki_page mp
     INNER JOIN relevant_wikis wp
           ON (mp.wiki_db = wp.wiki_db)
     WHERE page_namespace = 14
           AND snapshot = '2024-06'
),
redirects AS (
    SELECT mr.rd_from AS rd_from,
           tti.page_id AS rd_to,
           mr.wiki_db AS wiki_db
      FROM wmf_raw.mediawiki_redirect mr
     INNER JOIN title_to_id tti
           ON (mr.rd_title = tti.page_title
               AND mr.wiki_db = tti.wiki_db)
     WHERE mr.snapshot = '2024-06'
           AND mr.rd_namespace = 14
),
categorylinks_reformatted AS (
    SELECT cl.cl_from AS pid_from,
           tti.page_id AS cl_to,
           cl.wiki_db AS wiki_db
      F

24/07/23 14:54:37 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [18]:
spark.sql(f"select country, count(1) as num_pages from {category_results_table} group by country order by num_pages DESC").show(500, False)

+---------------------------------------------+---------+
|country                                      |num_pages|
+---------------------------------------------+---------+
|United States                                |4567976  |
|Germany                                      |1347638  |
|Japan                                        |1303868  |
|France                                       |1102911  |
|Italy                                        |1090208  |
|Spain                                        |867514   |
|United Kingdom                               |815701   |
|Australia                                    |730723   |
|Poland                                       |724064   |
|Canada                                       |704007   |
|Russia                                       |655270   |
|Brazil                                       |564582   |
|Switzerland                                  |531915   |
|South Korea                                  |462282   |
|Austria      

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 49850)
Traceback (most recent call last):
  File "/home/isaacj/.conda/envs/2024-04-29T19.33.29_isaacj/lib/python3.10/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/home/isaacj/.conda/envs/2024-04-29T19.33.29_isaacj/lib/python3.10/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/home/isaacj/.conda/envs/2024-04-29T19.33.29_isaacj/lib/python3.10/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/home/isaacj/.conda/envs/2024-04-29T19.33.29_isaacj/lib/python3.10/socketserver.py", line 747, in __init__
    self.handle()
  File "/home/isaacj/.conda/envs/2024-04-29T19.33.29_isaacj/lib/python3.10/site-packages/pyspark/accumulators.py", line 262, in handle
    poll(accum_updates)
  File "/home/isaacj/.cond